# Precision & Memory — see the formats yourself

A short companion to the lesson [*Precision & Memory*](https://lms-p-45c03.web.app/topics/math-infra/precision-and-memory/).
Five cells: cast a number into each float format and watch the **byte width**, the **range** (where fp16
overflows but bf16 holds), the **precision** (bf16 is coarser), and **fp4**'s tiny world.

**Runs on CPU** — just `numpy` + `ml_dtypes` (a JAX dependency Colab already has). No calculus, no GPU.

## 1. Setup

In [ ]:
# Colab has these; if not:  !pip install -q ml_dtypes
import numpy as np, ml_dtypes
np.seterr(over="ignore")            # we'll trigger overflow on purpose; don't shout about it

bf16 = ml_dtypes.bfloat16
fp4  = ml_dtypes.float4_e2m1fn      # OCP "E2M1": 1 sign + 2 exponent + 1 mantissa bit
print("numpy", np.__version__, "· ml_dtypes", ml_dtypes.__version__)

## 2. Bytes are the lever
A number's byte width is what it costs to move on the memory bus. Halving it ~doubles **arithmetic
intensity** (FLOPs per byte) — which is *why* lower precision keeps the matrix unit fed.

In [ ]:
for name, dt in [("fp32", np.float32), ("bf16", bf16), ("fp16", np.float16)]:
    print(f"{name}: {np.dtype(dt).itemsize} bytes / number")
print("fp4 : 0.5 bytes / number  (4 bits; numpy unpacks it to 1 byte)")

## 3. Range — why bf16 over fp16
bf16 keeps fp32's **8-bit exponent** (same reach); fp16 spends bits on precision and keeps only a
**5-bit exponent**, so it overflows past ~65,504. Cast a big activation and watch.

In [ ]:
def maxfinite(dt):
    try:    return float(ml_dtypes.finfo(dt).max)   # bf16 / fp4
    except Exception: return float(np.finfo(dt).max)  # fp16 / fp32

big = np.float32(1e5)               # a big activation value
print("cast 1e5 ->   bf16:", float(bf16(big)), "  |  fp16:", float(np.float16(big)))
print("max finite ->  fp16:", maxfinite(np.float16),
      " | bf16:", f"{maxfinite(bf16):.1e}", " | fp4:", maxfinite(fp4))
print("\nfp16 caps at 65,504 -> 1e5 overflows to inf. bf16 kept fp32's range (~3.4e38), so it holds.")

## 4. Precision — bf16 is coarser, and that's fine
Fewer mantissa bits = bigger rounding steps. bf16 rounds more than fp16 — but training averages that
out over millions of updates; what it can't lose is *range*.

In [ ]:
x = 0.1
print(f"representing {x}:")
for name, dt in [("fp32", np.float32), ("fp16", np.float16), ("bf16", bf16), ("fp4 ", fp4)]:
    r = float(dt(x))
    print(f"  {name}: {r:<12.6f} error {abs(r - x) / x * 100:5.1f}%")

## 5. fp4 (E2M1) — the extreme
Four bits buys ~2× throughput but a tiny world: one mantissa bit and a max of 6. It snaps every value
to a handful of magnitudes — which is why fp4 only works after values are **scaled into range**.

In [ ]:
representable = sorted({float(fp4(v)) for v in np.arange(0, 6.01, 0.05)})
print("fp4 can ONLY represent:", representable)
print("\nso real values snap hard:")
for v in [0.1, 1.7, 5.0, 7.0]:
    print(f"  {v}  ->  {float(fp4(v))}")
print("\n0.1 -> 0 (underflow), 7 -> 6 (saturates, E2M1 has no inf). Hence: scale first, then cast.")

## Put it together
1. **Bytes (§2):** going fp32 → bf16 halves the bytes. What does that do to arithmetic intensity, and why does it keep the MXU busy?
2. **Range (§3):** you saw 1e5 overflow fp16 but not bf16. Which bits made the difference?
3. **Precision (§4):** bf16's error on 0.1 is bigger than fp16's — so why does training prefer bf16 anyway?
4. **fp4 (§5):** why can't you just cast raw model values to fp4 — what has to happen first?

Back to the lesson → [Precision & Memory](https://lms-p-45c03.web.app/topics/math-infra/precision-and-memory/)